# UWF Zeek: fine-tuned LLM (Model 3)

Teaches Qwen2.5-7B-Instruct to read one 5-minute connection summary and give Sai's three answers:
`attack` (binary_target), `tactics` (tactic_targets) and `techniques` (mitre_targets).

- **Data:** `sai_edges_5min_capture.parquet` from the evaluation notebook (section 11): Sai's features, her three
  answers and the dataset each connection came from
- **Text:** plain-English summary of the connection plus what its source did in the same window; no IPs, times,
  dataset names or label columns
- **Split:** `SPLIT` switch. Default `leave_one_dataset_out` (train on 5 UWF datasets, test on the 6th), same as the GNN
- **Before / after:** the untrained model is scored first (comparable with Model 2), then fine-tuned with QLoRA and scored again
- **Test sampling:** all attack combinations and normal traffic are sampled with weights, so detection scores reflect
  the held-out dataset's real attack rate

Run order: setup → settings → data → split → prompts → scoring → model → zero-shot → fine-tune → fine-tuned scores.
Start with `PILOT = True` (small and fast) to check everything end to end.

## 1. Setup

In [ ]:
import importlib.util, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if importlib.util.find_spec("unsloth") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "unsloth"])

## 2. Settings

In [ ]:
import json, os, random, re, time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
try:
    display
except NameError:
    display = print

DRIVE_BASE = Path("/content/drive/Shareddrives/298B") if IN_COLAB else Path("./uwf_work")
CFG = {
    "SOURCE_FILE": str(DRIVE_BASE / "blocked_cv" / "sai_edges_5min_capture.parquet"),
    "OUT_DIR": str(DRIVE_BASE / "llm_finetune"),
    # split: "leave_one_dataset_out" (main), "year" (hold out a year group), "random_window" (Sai's random 80/10/10 windows)
    "SPLIT": "leave_one_dataset_out",
    "HOLD_OUT": "UWF-ZeekDataSum25-1",       # dataset (or "2021-22" / "2024" / "2025" for SPLIT="year")
    "PILOT": True,                          # small and fast end-to-end check
    "SEED": 42,
    # training set: cap per technique combination, the same number of normal connections, duplicates removed
    "TRAIN_CAP_PER_COMBO": 2000,
    "NORMAL_PER_ATTACK": 1.0,
    "VAL_FRACTION": 0.1,
    # test set: up to this many per attack combination, plus this many normal connections (weighted back to real counts)
    "TEST_CAP_PER_COMBO": 1000,
    "TEST_NORMAL": 5000,
    # model and training
    "BASE_MODEL": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    "MAX_SEQ_LEN": 1024,
    "LORA_RANK": 16,
    "EPOCHS": 2,
    "LEARNING_RATE": 2e-4,
    "BATCH_SIZE": 8,
    "GRAD_ACCUM": 2,
    "INFER_BATCH": 32,
    "RUN_MODEL": IN_COLAB,                  # load the LLM and run zero-shot, fine-tuning and scoring
}
CFG.update(json.loads(os.environ.get("UWF_LLM_OVERRIDES", "{}")))
if CFG["PILOT"]:
    CFG.update({"TRAIN_CAP_PER_COMBO": 150, "TEST_CAP_PER_COMBO": 150, "TEST_NORMAL": 600, "EPOCHS": 1})

OUT_DIR = Path(CFG["OUT_DIR"]) / f"{CFG['SPLIT']}__{CFG['HOLD_OUT']}{'__pilot' if CFG['PILOT'] else ''}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
print("source:", CFG["SOURCE_FILE"]); print("output:", OUT_DIR)

## 3. Connections as text

In [ ]:
edges = pd.read_parquet(CFG["SOURCE_FILE"])
edges["window_start"] = pd.to_datetime(edges["window_start"])


def as_list(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    return sorted(str(v) for v in value)


edges["tactics"] = edges["tactic_targets"].map(as_list)
edges["techniques"] = edges["mitre_targets"].map(as_list)
edges["attack"] = edges["binary_target"].astype(int) == 1
edges["year_group"] = edges["window_start"].dt.year.map({2021: "2021-22", 2022: "2021-22", 2024: "2024", 2025: "2025"})

## what the source and destination did in the same window (the context the GNN gets from its graph)
window_key = ["capture", "window_start"]
src_stats = edges.groupby(window_key + ["src_ip_zeek"]).agg(
    src_targets=("dest_ip_zeek", "nunique"), src_connections=("connection_count", "sum"),
    src_ports=("unique_destination_ports", "sum"))
dst_stats = edges.groupby(window_key + ["dest_ip_zeek"]).agg(dst_sources=("src_ip_zeek", "nunique"))
edges = edges.join(src_stats, on=window_key + ["src_ip_zeek"]).join(dst_stats, on=window_key + ["dest_ip_zeek"])

CONN_STATES = {
    "S0": "no reply to the connection attempt", "REJ": "connection rejected", "SF": "normal open and close",
    "RSTO": "reset by the originator", "RSTR": "reset by the responder", "RSTOS0": "originator sent SYN then reset",
    "RSTRH": "responder sent SYN-ACK then reset", "SH": "originator sent SYN then FIN", "SHR": "responder sent SYN-ACK then FIN",
    "S1": "established, not closed", "S2": "established, originator tried to close", "S3": "established, responder tried to close",
    "OTH": "no handshake seen",
}


def number(value):
    value = float(value or 0)
    return f"{value:,.0f}" if value >= 10 or value == int(value) else f"{value:.2f}"


def connection_text(row) -> str:
    service = row["most_common_service"] if isinstance(row["most_common_service"], str) and row["most_common_service"] else "unknown"
    state = row["most_common_connection_state"] if isinstance(row["most_common_connection_state"], str) else "unknown"
    port = row["most_common_destination_port"]
    port = "unknown" if port is None or (isinstance(port, float) and np.isnan(port)) else f"{int(port)}"
    return "\n".join([
        "Connection summary for one source-destination pair over 5 minutes:",
        f"- connections: {number(row['connection_count'])} (TCP {number(row['tcp_count'])}, UDP {number(row['udp_count'])}, ICMP {number(row['icmp_count'])})",
        f"- bytes sent: {number(row['total_orig_bytes'])}, bytes received: {number(row['total_resp_bytes'])}",
        f"- packets sent: {number(row['total_orig_packets'])}, packets received: {number(row['total_resp_packets'])}",
        f"- IP bytes sent: {number(row['total_orig_ip_bytes'])}, received: {number(row['total_resp_ip_bytes'])}, missed bytes: {number(row['total_missed_bytes'])}",
        f"- duration: total {number(row['total_duration'])} s, average {number(row['average_duration'])} s, longest {number(row['maximum_duration'])} s",
        f"- most common destination port: {port}; distinct destination ports: {number(row['unique_destination_ports'])}",
        f"- most common service: {service}",
        f"- most common connection state: {state} ({CONN_STATES.get(state, 'unknown state')})",
        f"- same 5 minutes, source overall: {number(row['src_targets'])} hosts contacted, {number(row['src_connections'])} connections",
        f"- same 5 minutes, destination overall: contacted by {number(row['dst_sources'])} sources",
    ])


def answer_json(row) -> str:
    return json.dumps({"attack": bool(row["attack"]), "tactics": row["tactics"], "techniques": row["techniques"]})


edges["text"] = [connection_text(r) for r in edges.to_dict("records")]
edges["answer"] = [answer_json(r) for r in edges.to_dict("records")]
edges["combo"] = np.where(edges["attack"],
                          [("T:" + ",".join(t)) if t else ("tactic-only:" + ",".join(a)) for t, a in zip(edges.techniques, edges.tactics)],
                          "normal")
print(f"{len(edges):,} connections | attacks: {int(edges.attack.sum()):,} | datasets: {edges.capture.nunique()}")
print("\nExample attack:\n" + edges.loc[edges.attack].iloc[0]["text"] + "\n→ " + edges.loc[edges.attack].iloc[0]["answer"])
print("\nExample normal:\n" + edges.loc[~edges.attack].iloc[0]["text"] + "\n→ " + edges.loc[~edges.attack].iloc[0]["answer"])

## 4. Split, training set and weighted test set

In [ ]:
from sklearn.model_selection import train_test_split


def split_masks(df):
    """Returns (train_pool, test) boolean masks for the chosen SPLIT."""
    if CFG["SPLIT"] == "leave_one_dataset_out":
        test = (df.capture == CFG["HOLD_OUT"]).to_numpy()
    elif CFG["SPLIT"] == "year":
        test = (df.year_group == CFG["HOLD_OUT"]).to_numpy()
    elif CFG["SPLIT"] == "random_window":
        ## Sai's split: whole 5-minute windows shuffled 80/10/10, stratified by whether the window has an attack
        windows = df.groupby("window_start").attack.max().reset_index()
        train_w, rest_w = train_test_split(windows, test_size=0.2, random_state=CFG["SEED"], stratify=windows.attack)
        _, test_w = train_test_split(rest_w, test_size=0.5, random_state=CFG["SEED"], stratify=rest_w.attack)
        test = df.window_start.isin(set(test_w.window_start)).to_numpy()
        return df.window_start.isin(set(train_w.window_start)).to_numpy(), test
    else:
        raise ValueError(CFG["SPLIT"])
    return ~test, test


train_pool_mask, test_mask = split_masks(edges)
assert test_mask.any(), f"nothing to test: check HOLD_OUT={CFG['HOLD_OUT']!r}"
pool, test_pool = edges[train_pool_mask], edges[test_mask]

## training set: cap each technique combination, add normal traffic, drop duplicate texts (keep the majority answer)
rng = np.random.default_rng(CFG["SEED"])
attack_parts = [g.sample(min(len(g), CFG["TRAIN_CAP_PER_COMBO"]), random_state=CFG["SEED"])
                for _, g in pool[pool.attack].groupby("combo")]
attacks = pd.concat(attack_parts) if attack_parts else pool.iloc[0:0]
normal_pool = pool[~pool.attack]
normals = normal_pool.sample(min(len(normal_pool), int(len(attacks) * CFG["NORMAL_PER_ATTACK"]) or 1), random_state=CFG["SEED"])
train_all = pd.concat([attacks, normals])
majority = train_all.groupby("text").answer.agg(lambda s: s.value_counts().index[0])
train_all = train_all.drop_duplicates("text").assign(answer=lambda d: d.text.map(majority))
train_all = train_all.sample(frac=1.0, random_state=CFG["SEED"])
n_val = max(1, int(len(train_all) * CFG["VAL_FRACTION"]))
val_set, train_set = train_all.iloc[:n_val], train_all.iloc[n_val:]

## test set: sample each stratum (attack combination or normal) and keep a weight = stratum size / sample size
test_parts = []
for combo, group in test_pool.groupby("combo"):
    cap = CFG["TEST_NORMAL"] if combo == "normal" else CFG["TEST_CAP_PER_COMBO"]
    sample = group.sample(min(len(group), cap), random_state=CFG["SEED"])
    test_parts.append(sample.assign(weight=len(group) / len(sample)))
test_set = pd.concat(test_parts)

## techniques the model can learn: present in the training pool
learnable = set(t for ts in pool.loc[pool.attack, "techniques"] for t in ts)
summary = pd.DataFrame({
    "connections": [len(train_set), len(val_set), len(test_set)],
    "attacks": [int(train_set.attack.sum()), int(val_set.attack.sum()), int(test_set.attack.sum())],
    "represents (weighted)": [len(train_set), len(val_set), int(test_set.weight.sum())],
}, index=["train", "validation", "test"])
display(summary)
print("attack combinations in training:", train_set[train_set.attack].combo.nunique(),
      "| techniques in held-out data:", sorted(set(t for ts in test_pool.techniques for t in ts)),
      "| of which learnable:", sorted(set(t for ts in test_pool.techniques for t in ts) & learnable))
for name, frame in [("train", train_set), ("val", val_set), ("test", test_set)]:
    frame.drop(columns=["tactic_targets", "mitre_targets"]).to_parquet(OUT_DIR / f"{name}.parquet", index=False)

## 5. Prompt and answer format

In [ ]:
TACTICS = sorted(set(t for ts in edges.tactics for t in ts))
TECHNIQUES = sorted(set(t for ts in edges.techniques for t in ts))
TECHNIQUE_NAMES = {
    "T1016": "System Network Configuration Discovery", "T1018": "Remote System Discovery", "T1046": "Network Service Discovery",
    "T1048": "Exfiltration Over Alternative Protocol", "T1059": "Command and Scripting Interpreter", "T1071": "Application Layer Protocol",
    "T1078": "Valid Accounts", "T1110": "Brute Force", "T1112": "Modify Registry", "T1133": "External Remote Services",
    "T1136": "Create Account", "T1190": "Exploit Public-Facing Application", "T1203": "Exploitation for Client Execution",
    "T1204": "User Execution", "T1210": "Exploitation of Remote Services", "T1505": "Server Software Component",
    "T1546": "Event Triggered Execution", "T1547": "Boot or Logon Autostart Execution", "T1548": "Abuse Elevation Control Mechanism",
    "T1557": "Adversary-in-the-Middle", "T1566": "Phishing", "T1571": "Non-Standard Port", "T1587": "Develop Capabilities",
    "T1589": "Gather Victim Identity Information", "T1590": "Gather Victim Network Information",
    "T1592": "Gather Victim Host Information", "T1595": "Active Scanning",
}
SYSTEM_PROMPT = (
    "You are a network security analyst. You read a summary of the Zeek connection logs between one source and one "
    "destination over 5 minutes and decide whether it is part of an attack.\n"
    "Answer with JSON only, in this exact form: "
    '{"attack": true or false, "tactics": [...], "techniques": [...]}.\n'
    "For normal traffic both lists are empty. An attack can have more than one tactic and technique; list technique "
    "IDs only, and leave techniques empty if none applies.\n"
    f"Allowed tactics: {', '.join(TACTICS)}.\n"
    "Allowed techniques: " + "; ".join(f"{t} {TECHNIQUE_NAMES.get(t, '')}".strip() for t in TECHNIQUES) + "."
)
ANSWER_PREFIX = '{"attack":'


def messages(text, answer=None):
    turns = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}]
    if answer is not None:
        turns.append({"role": "assistant", "content": answer})
    return turns


def parse_answer(raw: str) -> dict:
    """Reads the model's JSON; anything outside the allowed labels is kept separately as invalid."""
    match = re.search(r"\{.*\}", raw or "", flags=re.S)
    result = {"attack": None, "tactics": [], "techniques": [], "invalid": [], "parsed": False}
    if not match:
        return result
    try:
        payload = json.loads(match.group(0))
    except json.JSONDecodeError:
        return result
    result["parsed"] = True
    result["attack"] = bool(payload.get("attack")) if isinstance(payload.get("attack"), bool) else None
    for key, allowed in [("tactics", TACTICS), ("techniques", TECHNIQUES)]:
        values = payload.get(key) or []
        values = values if isinstance(values, list) else [values]
        result[key] = sorted({str(v) for v in values if str(v) in allowed})
        result["invalid"] += [str(v) for v in values if str(v) not in allowed]
    return result


print(SYSTEM_PROMPT[:600] + " ...")
print(f"\nprompt length ≈ {len(SYSTEM_PROMPT) // 4 + 250} tokens")

## 6. Scoring

In [ ]:
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score


def score_predictions(pred: pd.DataFrame, frame: pd.DataFrame) -> dict:
    """pred: index aligned with frame; columns p_attack, attack_pred, tactics_pred, techniques_pred, parsed, invalid."""
    w = frame["weight"].to_numpy() if "weight" in frame else np.ones(len(frame))
    y = frame["attack"].to_numpy().astype(int)
    p = pred["p_attack"].to_numpy()
    yhat = pred["attack_pred"].fillna(False).to_numpy().astype(bool)
    tp, fp = float(w[yhat & (y == 1)].sum()), float(w[yhat & (y == 0)].sum())
    fn, tn = float(w[~yhat & (y == 1)].sum()), float(w[~yhat & (y == 0)].sum())
    scores = {
        "parsed_answers": float(pred["parsed"].mean()),
        "answers_with_invalid_labels": float((pred["invalid"].map(len) > 0).mean()),
        "attack_pr_auc": float(average_precision_score(y, p, sample_weight=w)) if 0 < y.sum() < len(y) else np.nan,
        "attack_roc_auc": float(roc_auc_score(y, p, sample_weight=w)) if 0 < y.sum() < len(y) else np.nan,
        "attack_recall": tp / max(tp + fn, 1e-9), "attack_precision": tp / max(tp + fp, 1e-9),
        "false_alarm_rate": fp / max(fp + tn, 1e-9),
    }
    attacks = frame["attack"].to_numpy()
    for key, classes in [("tactics", TACTICS), ("techniques", TECHNIQUES)]:
        true_lists = frame.loc[attacks, key].tolist()
        pred_lists = pred.loc[attacks, f"{key}_pred"].tolist()
        keep = [i for i, t in enumerate(true_lists) if t]
        if not keep:
            continue
        index = {c: i for i, c in enumerate(classes)}
        Y = np.zeros((len(keep), len(classes))); P = np.zeros_like(Y)
        for row, i in enumerate(keep):
            for c in true_lists[i]: Y[row, index[c]] = 1
            for c in pred_lists[i]: P[row, index[c]] = 1
        present = Y.sum(0) > 0
        scores[f"{key}_exact_match"] = float(np.mean([set(true_lists[i]) == set(pred_lists[i]) for i in keep]))
        scores[f"{key}_first_correct"] = float(np.mean([bool(pred_lists[i]) and pred_lists[i][0] in true_lists[i] for i in keep]))
        scores[f"{key}_micro_f1"] = float(f1_score(Y, P, average="micro", zero_division=0))
        scores[f"{key}_macro_f1"] = float(f1_score(Y[:, present], P[:, present], average="macro", zero_division=0))
    return scores


def per_technique(pred: pd.DataFrame, frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    attacks = frame["attack"].to_numpy()
    for technique in sorted(set(t for ts in frame.loc[attacks, "techniques"] for t in ts)):
        mask = frame["techniques"].map(lambda ts: technique in ts).to_numpy() & attacks
        found = pred.loc[mask, "techniques_pred"].map(lambda ts: technique in ts)
        rows.append({"technique": technique, "test_connections": int(mask.sum()),
                     "learnable (in training data)": technique in learnable, "recall": float(found.mean())})
    return pd.DataFrame(rows).sort_values("test_connections", ascending=False)

## 7. Load the base model (Qwen2.5-7B-Instruct, 4-bit)

In [ ]:
if CFG["RUN_MODEL"]:
    import torch
    from unsloth import FastLanguageModel
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()        # hides the harmless "max_new_tokens and max_length" warning on every batch

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=CFG["BASE_MODEL"], max_seq_length=CFG["MAX_SEQ_LEN"], load_in_4bit=True, dtype=None)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    TRUE_ID = tokenizer.encode(" true", add_special_tokens=False)[0]
    FALSE_ID = tokenizer.encode(" false", add_special_tokens=False)[0]
    print("loaded", CFG["BASE_MODEL"], "| true/false token ids:", TRUE_ID, FALSE_ID)


def run_model(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    """Greedy answers for every row; p_attack = P(' true') vs P(' false') right after '{"attack":'."""
    out_file = OUT_DIR / f"predictions_{label}.parquet"
    if out_file.exists():
        return pd.read_parquet(out_file).set_index("row")
    has_add_on = any("lora_" in name for name, _ in model.named_parameters())
    assert label != "zero_shot" or not has_add_on, "zero-shot needs the untrained model: restart the runtime and run sections 1-8 only"
    FastLanguageModel.for_inference(model)
    texts = frame["text"].tolist()
    unique = list(dict.fromkeys(texts))                     # score each distinct text once
    results = {}
    started = time.time()
    for start in range(0, len(unique), CFG["INFER_BATCH"]):
        batch = unique[start:start + CFG["INFER_BATCH"]]
        prompts = [tokenizer.apply_chat_template(messages(t), tokenize=False, add_generation_prompt=True) + ANSWER_PREFIX
                   for t in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=96, do_sample=False, output_scores=True,
                                       return_dict_in_generate=True, pad_token_id=tokenizer.pad_token_id)
        first = torch.softmax(generated.scores[0][:, [TRUE_ID, FALSE_ID]].float(), dim=-1)[:, 0].cpu().numpy()
        new_tokens = generated.sequences[:, inputs["input_ids"].shape[1]:]
        for text, prob, tokens in zip(batch, first, new_tokens):
            raw = ANSWER_PREFIX + tokenizer.decode(tokens, skip_special_tokens=True)
            results[text] = (float(prob), raw)
        if start // CFG["INFER_BATCH"] % 20 == 0:
            print(f"  {label}: {start + len(batch):,}/{len(unique):,} distinct texts ({(time.time() - started) / 60:.1f} min)")
    rows = []
    for row, text in zip(frame.index, texts):
        prob, raw = results[text]
        parsed = parse_answer(raw)
        rows.append({"row": row, "p_attack": prob,
                     "attack_pred": parsed["attack"] if parsed["attack"] is not None else prob >= 0.5,
                     "tactics_pred": parsed["tactics"], "techniques_pred": parsed["techniques"],
                     "parsed": parsed["parsed"], "invalid": parsed["invalid"], "raw": raw})
    pred = pd.DataFrame(rows).set_index("row")
    pred.reset_index().to_parquet(out_file, index=False)
    return pred

## 8. Before fine-tuning: zero-shot scores

In [ ]:
if CFG["RUN_MODEL"]:
    zero_pred = run_model(test_set, "zero_shot")
    zero_scores = score_predictions(zero_pred, test_set)
    json.dump(zero_scores, open(OUT_DIR / "scores_zero_shot.json", "w"), indent=1)
    display(pd.Series(zero_scores).round(4))

## 9. Fine-tune (QLoRA: compressed base model + small trainable add-on)

In [ ]:
ADAPTER_DIR = OUT_DIR / "adapter"

if CFG["RUN_MODEL"] and not (ADAPTER_DIR / "adapter_config.json").exists():
    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer
    from unsloth.chat_templates import train_on_responses_only

    model = FastLanguageModel.get_peft_model(
        model, r=CFG["LORA_RANK"], lora_alpha=CFG["LORA_RANK"], lora_dropout=0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=CFG["SEED"])
    tokenizer.padding_side = "right"

    def to_dataset(frame):
        texts = [tokenizer.apply_chat_template(messages(t, a), tokenize=False) for t, a in zip(frame.text, frame.answer)]
        return Dataset.from_dict({"text": texts})

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=to_dataset(train_set), eval_dataset=to_dataset(val_set),
        args=SFTConfig(
            dataset_text_field="text", max_seq_length=CFG["MAX_SEQ_LEN"],
            per_device_train_batch_size=CFG["BATCH_SIZE"], gradient_accumulation_steps=CFG["GRAD_ACCUM"],
            num_train_epochs=CFG["EPOCHS"], learning_rate=CFG["LEARNING_RATE"], warmup_ratio=0.03,
            lr_scheduler_type="linear", optim="adamw_8bit", weight_decay=0.01,
            logging_steps=20, eval_strategy="steps", eval_steps=200, save_strategy="no",
            output_dir=str(OUT_DIR / "trainer"), seed=CFG["SEED"], report_to="none"),
    )
    ## learn only from the answer, not from the prompt
    trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                      response_part="<|im_start|>assistant\n")
    started = time.time()
    trainer.train()
    print(f"training took {(time.time() - started) / 60:.1f} min")
    model.save_pretrained(str(ADAPTER_DIR)); tokenizer.save_pretrained(str(ADAPTER_DIR))
    pd.DataFrame(trainer.state.log_history).to_csv(OUT_DIR / "training_log.csv", index=False)
    tokenizer.padding_side = "left"
elif CFG["RUN_MODEL"]:
    ## already trained in an earlier session: load base model + saved add-on
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(ADAPTER_DIR), max_seq_length=CFG["MAX_SEQ_LEN"], load_in_4bit=True, dtype=None)
    tokenizer.padding_side = "left"
    print("loaded saved adapter from", ADAPTER_DIR)

## 10. After fine-tuning: scores and comparison

In [ ]:
if CFG["RUN_MODEL"]:
    tuned_pred = run_model(test_set, "fine_tuned")
    tuned_scores = score_predictions(tuned_pred, test_set)
    json.dump(tuned_scores, open(OUT_DIR / "scores_fine_tuned.json", "w"), indent=1)

    comparison = pd.DataFrame({"zero-shot": zero_scores, "fine-tuned": tuned_scores})
    print(f"=== {CFG['SPLIT']} | held out: {CFG['HOLD_OUT']} | pilot: {CFG['PILOT']} ===")
    display(comparison.round(4))
    print("\nper technique (fine-tuned):")
    display(per_technique(tuned_pred, test_set).round(4))
    comparison.to_csv(OUT_DIR / "comparison.csv")
    print("\nexample answers:")
    for i in test_set.index[:3]:
        print("  true:", test_set.loc[i, "answer"], "\n  pred:", tuned_pred.loc[i, "raw"][:160])

## 11. Next
- Pilot looks right (answers parse, scores move from zero-shot)? Set `PILOT = False` and rerun from section 2.
- Other folds: change `HOLD_OUT` (start with `UWF-ZeekData24` and `UWF-ZeekDataFall24-2`), rerun from section 2.
  Each fold saves to its own folder under `llm_finetune/`, and finished steps are skipped on rerun.
- Compare with the GNN on the same held-out dataset (evaluation notebook, sections 9c and 11.6).
- Model 4: serve the saved adapter behind an OpenAI-compatible server and point the backend at it (`LLM_PROVIDER=local`).